In [24]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score

In [25]:
df= pd.read_csv(r"E:\python\.newvenv\Data\used_car_price_prediction_Data.csv")

In [3]:
df

,Brand,Model,Year,Mileage_kmpl,Engine_CC,Horsepower,Fuel_Type,Transmission,Owner_Type,Color,City,Kms_Driven,Insurance_Valid,Service_History,Accidents,Tax_Paid,Number_of_Doors,Seats,Registration_Age,Price
0,Toyota,Camry,2001,19.615852,1396.560379,NaN,Hybrid,Automatic,First,Grey,Chennai,500928,1,1,0,1,4,5,24,3.531896e+05
1,Hyundai,i20,2012,19.478608,1130.771005,137.021719,Petrol,NaN,First,Grey,Mumbai,211510,0,1,0,1,4,5,13,9.596943e+05
2,Mahindra,Bolero,2008,17.469920,1766.466250,142.902960,NaN,Automatic,First,Black,Kolkata,333999,1,0,0,1,5,5,17,2.920229e+05
3,BMW,X1,2015,21.295421,NaN,49.782079,NaN,Manual,First,White,Pune,225930,1,1,3,1,4,5,10,2.949453e+06
4,Honda,Civic,2022,18.326833,1239.334935,121.505993,Diesel,Manual,Second,White,Kolkata,43404,1,0,0,1,5,5,3,6.365208e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1004995,Nissan,Magnite,2022,12.256422,1773.657889,117.574596,Petrol,Manual,First,Black,Ahmedabad,68571,1,0,0,1,4,5,3,1.104233e+06
1004996,Honda,Amaze,2005,NaN,1705.773023,99.643689,CNG,Automatic,First,NaN,Pune,311400,1,1,0,1,4,5,20,7.122346e+05
1004997,Skoda,Octavia,2018,21.665335,1447.159515,NaN,Petrol,NaN,First,Brown,Delhi,42140,0,1,0,1,4,4,7,1.304198e+06
1004998,BMW,X3,2010,17.852635,2179.906631,NaN,Petrol,Manual,First,Brown,Mumbai,211980,0,1,0,1,4,7,15,1.866610e+06


In [5]:
df.shape

(1005000, 20)

In [6]:
df = df.dropna()

In [7]:
df.shape

(561457, 20)

In [26]:
import numpy as np

# Clean the dataframe

df = df.copy()

# Remove duplicate rows and rows without a target value
df = df.drop_duplicates().reset_index(drop=True)
df = df.dropna(subset=['Price']).copy()

# Convert price and common numeric columns to numeric
numeric_cols = ['Price', 'Mileage_kmpl', 'Engine_CC', 'Horsepower', 'Seats', 'Number_of_Doors']
for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(r'[^0-9.\-]+', '', regex=True),
        errors='coerce'
    )

# Fill missing values intelligently
for col in df.columns:
    if col == 'Price':
        continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna('Unknown').astype(str)

# One-hot encode categorical columns
cat_cols = [c for c in df.columns if c != 'Price' and not pd.api.types.is_numeric_dtype(df[c])]
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Final cleanup
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str)

df = df.replace([np.inf, -np.inf], np.nan).dropna()
df.head()

,Year,Mileage_kmpl,Engine_CC,Horsepower,Kms_Driven,Insurance_Valid,Service_History,Accidents,Tax_Paid,Number_of_Doors,...,Color_Unknown,Color_White,City_Bangalore,City_Chennai,City_Delhi,City_Hyderabad,City_Kolkata,City_Mumbai,City_Pune,City_Unknown
0,2001,19.615852,1396.560379,120.057888,500928,1,1,0,1,4,...,False,False,False,True,False,False,False,False,False,False
1,2012,19.478608,1130.771005,137.021719,211510,0,1,0,1,4,...,False,False,False,False,False,False,False,True,False,False
2,2008,17.469920,1766.466250,142.902960,333999,1,0,0,1,5,...,False,False,False,False,False,False,True,False,False,False
3,2015,21.295421,1500.202560,49.782079,225930,1,1,3,1,4,...,False,True,False,False,False,False,False,False,True,False
4,2022,18.326833,1239.334935,121.505993,43404,1,0,0,1,5,...,False,True,False,False,False,False,True,False,False,False


In [27]:
# Fill remaining numeric gaps with column medians
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())

# Keep the target as numeric and convert integer-like columns safely
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Convert only columns that are truly integer-like
for col in ['Seats', 'Number_of_Doors']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').round().astype('Int64')

# Drop any remaining rows with missing values after cleaning
df = df.dropna().reset_index(drop=True)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Columns: 107 entries, Year to City_Unknown
dtypes: Int64(2), bool(94), float64(4), int64(7)
memory usage: 190.7 MB


In [10]:
# Fill any remaining NaN values with column means for numeric columns
df = df.copy()
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].mean())
    else:
        df[col] = df[col].fillna('Unknown')


,Mileage_kmpl,Engine_CC,Horsepower,Number_of_Doors,Seats,Price
4,18.326833,1239.334935,121.505993,5,5,6.365208e+05
7,21.895120,800.000000,0.709766,5,5,5.000000e+04
8,17.461847,1397.891768,104.095168,4,5,1.045347e+06
9,12.062221,1618.723936,86.071024,5,7,2.633309e+05
10,16.831229,2128.230552,146.219245,4,5,4.032569e+06
...,...,...,...,...,...,...
1004988,8.097582,1641.225144,118.952121,4,5,8.224758e+05
1004990,13.935683,1287.775569,114.498130,4,7,3.963039e+05
1004991,19.946978,1252.293439,115.394192,4,2,5.000000e+04
1004995,12.256422,1773.657889,117.574596,4,5,1.104233e+06


In [11]:
df.info()

<class 'pandas.DataFrame'>
Index: 561457 entries, 4 to 1004999
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   Mileage_kmpl     561457 non-null  float64
 1   Engine_CC        561457 non-null  float64
 2   Horsepower       561457 non-null  float64
 3   Number_of_Doors  561457 non-null  int64  
 4   Seats            561457 non-null  int64  
 5   Price            561457 non-null  float64
dtypes: float64(4), int64(2)
memory usage: 30.0 MB


In [12]:
df.isnull().sum()

Mileage_kmpl       0
Engine_CC          0
Horsepower         0
Number_of_Doors    0
Seats              0
Price              0
dtype: int64

In [33]:
# Keep the cleaned dataframe and preserve categorical columns for modeling
# Convert numeric-like columns safely
for col in ['Seats', 'Number_of_Doors']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')

# Fill any remaining missing values for numeric columns and use 'Unknown' for categoricals
df = df.copy()
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna('Unknown').astype(str)

# Convert boolean columns to integers so scikit-learn can process them
for col in df.columns:
    if pd.api.types.is_bool_dtype(df[col]):
        df[col] = df[col].astype('int64')

# Preview the cleaned dataset
df.head()

,Year,Mileage_kmpl,Engine_CC,Horsepower,Kms_Driven,Insurance_Valid,Service_History,Accidents,Tax_Paid,Number_of_Doors,...,Color_Unknown,Color_White,City_Bangalore,City_Chennai,City_Delhi,City_Hyderabad,City_Kolkata,City_Mumbai,City_Pune,City_Unknown
0,2001,19.615852,1396.560379,120.057888,500928,1,1,0,1,4.0,...,0,0,0,1,0,0,0,0,0,0
1,2012,19.478608,1130.771005,137.021719,211510,0,1,0,1,4.0,...,0,0,0,0,0,0,0,1,0,0
2,2008,17.469920,1766.466250,142.902960,333999,1,0,0,1,5.0,...,0,0,0,0,0,0,1,0,0,0
3,2015,21.295421,1500.202560,49.782079,225930,1,1,3,1,4.0,...,0,1,0,0,0,0,0,0,1,0
4,2022,18.326833,1239.334935,121.505993,43404,1,0,0,1,5.0,...,0,1,0,0,0,0,1,0,0,0


In [14]:
df.head()

,Mileage_kmpl,Engine_CC,Horsepower,Number_of_Doors,Seats,Price
4,18,1239,122,5,5,636521
7,22,800,1,5,5,50000
8,17,1398,104,4,5,1045347
9,12,1619,86,5,7,263331
10,17,2128,146,4,5,4032569


In [37]:
# Feature selection
X = df.drop('Price', axis=1)
y = df['Price']

In [16]:
X

,Mileage_kmpl,Engine_CC,Horsepower,Number_of_Doors,Seats
4,18,1239,122,5,5
7,22,800,1,5,5
8,17,1398,104,4,5
9,12,1619,86,5,7
10,17,2128,146,4,5
...,...,...,...,...,...
1004988,8,1641,119,4,5
1004990,14,1288,114,4,7
1004991,20,1252,115,4,2
1004995,12,1774,118,4,5


In [17]:
y

4           636521
7            50000
8          1045347
9           263331
10         4032569
            ...   
1004988     822476
1004990     396304
1004991      50000
1004995    1104233
1004999      50000
Name: Price, Length: 561457, dtype: int64

In [38]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [36]:
X_train.head()

,Year,Mileage_kmpl,Engine_CC,Horsepower,Kms_Driven,Insurance_Valid,Service_History,Accidents,Tax_Paid,Number_of_Doors,...,Color_Unknown,Color_White,City_Bangalore,City_Chennai,City_Delhi,City_Hyderabad,City_Kolkata,City_Mumbai,City_Pune,City_Unknown
566853,2016,15.198124,1587.774589,120.057888,87417,1,1,0,1,2.0,...,True,False,False,False,False,False,False,False,True,False
382311,2019,25.112967,800.000000,120.057888,121536,1,1,0,1,5.0,...,False,False,False,False,False,False,False,False,True,False
241519,2004,17.998460,2168.806244,196.162692,370881,1,1,1,1,2.0,...,False,False,False,False,False,False,True,False,False,False
719220,2015,17.998460,1457.213945,95.445216,197350,1,1,0,1,4.0,...,False,False,False,False,False,False,True,False,False,False
905718,2002,17.998460,1497.493436,141.909657,532887,1,1,0,1,5.0,...,False,False,False,False,False,False,False,False,False,True


In [39]:
# Build preprocessing pipeline for numeric + categorical features
numeric_features = X.select_dtypes(include=['number']).columns
categorical_features = X.select_dtypes(exclude=['number']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [ ]:
# Random forest regressor with preprocessing pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)

In [22]:
#predictions
y_pred = rf_pipeline.predict(X_test)
print("Predictions:", y_pred)

Predictions: [ 918988.85268849  941280.1283846  1089657.7926139  ...  998093.23188881
 1015271.16914737 1051006.92584084]


In [23]:
from sklearn.metrics import r2_score
print("Score",r2_score(y_test,y_pred))

Score 0.0014912019569612855


In [ ]:
#graiident boost
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
# Gradient boosting regressor with preprocessing pipeline
gb_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        min_samples_split=5,
        min_samples_leaf=2,
        subsample=0.8,
        random_state=42
    ))
])

In [ ]:
gb_pipeline.fit(X_train, y_train)

,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",0.8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",5
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, in

In [ ]:
pred = gb_pipeline.predict(X_test)

In [ ]:
pred

array([1056798.73201723,  942842.48419029, 1002359.05982918, ...,
       1007623.31934809, 1086737.40736891,  989671.65164501],
      shape=(201000,))

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
print("R2 Score :", r2_score(y_test, pred))

R2 Score : 0.0012863689327873296
